# Load Datasets to BigQuery
- May 2026
- Numantic Solutions (numanticsolutions.com)

In [2]:
import os, sys
import json

# Pandas and Pandas Google Big Query
import pandas as pd
import pandas_gbq as pbq

# Numantic utilities
utils_path = "../utils"
sys.path.insert(0, utils_path)
from utils import ApiAuthentication
api_configs = ApiAuthentication(client="Numantic")


## Read local document and test Q & A data


In [3]:
input_data_path = "../data/rag_eval_dataset"
docs_filename = "documents.csv"
no_answer_qs = "no_answer_questions.csv"
single_pas_answer_qs = "single_passage_answer_questions.csv"

# Read local data into Pandas dataframes
df_docs = pd.read_csv(filepath_or_buffer=os.path.join(input_data_path, docs_filename))
df_noaqs = pd.read_csv(filepath_or_buffer=os.path.join(input_data_path, no_answer_qs))
df_spqs = pd.read_csv(filepath_or_buffer=os.path.join(input_data_path, single_pas_answer_qs))

## Clean up docs

## Load some document metadata

In [4]:
doc_metadata_file ="doc_metadata.json"
data_path = "../data/rag_eval_dataset"

with open(os.path.join(data_path,doc_metadata_file), "r", encoding="utf-8") as file:
    doc_metadata = json.load(file)


## Add metadata attributes and supplement text as needed

In [5]:
# Metadata columns
df_docs["source_type"] = df_docs["source_url"].map(doc_metadata["source_type_map"])
df_docs["title"] = df_docs["source_url"].map(doc_metadata["source_title_map"])

# Missing text
for key in doc_metadata["text_additions"].keys():
    doc_index = int(key.replace("doc_", ""))
    df_docs.loc[doc_index, "text"] = "{}\n\n{}".format(df_docs.loc[1, "text"],
                                                       doc_metadata["text_additions"][key])


## Add and rename columns

In [6]:
# Add a document index column
df_docs["_id"] = df_docs["index"].apply(lambda x: "{}".format(x))
df_docs["doc_index"] = df_docs["index"].apply(lambda x: "doc_{}".format(x))
df_docs = df_docs.drop(columns="index")

# Rename and reorder columns
col_renm = {"text": "content"}
df_docs = df_docs.rename(columns=col_renm)
dfd_cols = [ '_id', 'doc_index', 'source_url', 'source_type', 'title', 'content']
df_docs = df_docs[dfd_cols]


In [7]:
df_docs.head()

,_id,doc_index,source_url,source_type,title,content
0,0,doc_0,https://enterthegungeon.fandom.com/wiki/Bullet...,gaming,Bullet Kin,Bullet Kin\nBullet Kin are one of the most com...
1,1,doc_1,https://www.dropbox.com/scl/fi/ljtdg6eaucrbf1a...,gaming,The Paths through the Underground/Underdark,---The Paths through the Underground/Underdark...
2,2,doc_2,https://bytes-and-nibbles.web.app/bytes/stici-...,data_science,Semantic and Textual Inference Chatbot Interfa...,Semantic and Textual Inference Chatbot Interfa...
3,3,doc_3,https://github.com/llmware-ai/llmware,data_science,LLMware,llmware\n\nBuilding Enterprise RAG Pipelines w...
4,4,doc_4,https://docs.marimo.io/recipes.html,recipes,Building Block Recipes,Recipes\nThis page includes code snippets or “...


## Create a passages dataframe

In [10]:
prows = []
pid = 1
for idx in df_docs.index:

    raw_text = df_docs.loc[idx, "content"]

    blocks = raw_text.split('\n\n')
    for block in blocks:
        prows.append(dict(_id=pid,
                          doc_index=df_docs.loc[idx, "doc_index"],
                          source_url=df_docs.loc[idx, "source_url"],
                          source_type=df_docs.loc[idx, "source_type"],
                          title=df_docs.loc[idx, "title"],
                          content=block
                          )
                     )
        pid += 1


df_pass = pd.DataFrame(data=prows)


In [11]:

df_pass.head()


,_id,doc_index,source_url,source_type,title,content
0,1,doc_0,https://enterthegungeon.fandom.com/wiki/Bullet...,gaming,Bullet Kin,Bullet Kin\nBullet Kin are one of the most com...
1,2,doc_0,https://enterthegungeon.fandom.com/wiki/Bullet...,gaming,Bullet Kin,"Occasionally, Bullet Kin will have assault rif..."
2,3,doc_0,https://enterthegungeon.fandom.com/wiki/Bullet...,gaming,Bullet Kin,On some occasions the player will also encount...
3,4,doc_0,https://enterthegungeon.fandom.com/wiki/Bullet...,gaming,Bullet Kin,"In the Black Powder Mine, they can also ride M..."
4,5,doc_0,https://enterthegungeon.fandom.com/wiki/Bullet...,gaming,Bullet Kin,Trivia\nBullet Kin wield Magnums. Assault-rifl...


## Load documents to BigQuery

In [12]:
# Load dataframe to BigQuery
dataset_id = "ns_bq"
project_id = os.environ["GOOGLE_CLOUD_PROJECT_ID"]

table_name = "{}.rag_tests_2".format(dataset_id)
pbq.to_gbq(dataframe=df_docs,
           destination_table=table_name,
           project_id=project_id,
           if_exists="replace",
           progress_bar="tqdm")

table_name = "{}.rag_tests_3".format(dataset_id)
pbq.to_gbq(dataframe=df_pass,
           destination_table=table_name,
           project_id=project_id,
           if_exists="replace",
           progress_bar="tqdm")

100%|██████████| 1/1 [00:00<00:00, 15592.21it/s]
